# TMC2130 live control

Sliders talk to the test-bench firmware (`src/main.cpp`) over the ST-Link serial port.
Speed changes are ramped by the firmware (`accel`), so dragging the rpm slider is safe.
Run the cells top to bottom; uncomment and run the last cell to free the port before flashing (flashing over ST-Link works while the port is open too).

In [5]:
%pip install -q pyserial ipywidgets


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import serial, serial.tools.list_ports

PORT = None  # e.g. '/dev/cu.usbmodem11203'; None = first ST-Link / USB modem port
PORT = PORT or next(p.device for p in serial.tools.list_ports.comports() if 'usbmodem' in p.device or 'STLink' in str(p.description))
ser = serial.Serial(PORT, 115200, timeout=0.1)

def send(cmd):
    """Send one command, return the firmware's status reply. Reopens the port if it was closed."""
    if not ser.is_open:
        ser.open()
    ser.reset_input_buffer()
    ser.write(f'{cmd}\n'.encode())
    time.sleep(0.08)
    return ser.read(2000).decode(errors='replace').strip()

print(PORT)
print(send('?'))

In [ ]:
import ipywidgets as w

status = w.Output(layout={'border': '1px solid #ccc', 'padding': '4px'})
def show(text):
    with status:
        status.clear_output()
        print(text)

def run(cmd):
    try:
        show(send(cmd))
    except Exception as e:  # widget callbacks swallow errors, so show them here
        show(f'ERROR {cmd!r}: {e}')

controls = []  # (widget, command letter)
def ctl(widget, letter):
    widget.observe(lambda ch: run(f'{letter}{ch.new:g}'), names='value')
    controls.append((widget, letter))
    return widget

wide = dict(layout=w.Layout(width='650px'), style={'description_width': '110px'})
rpm   = ctl(w.FloatSlider(value=0, min=-800, max=800, step=5, description='speed rpm', **wide), 'v')
mstep = ctl(w.SelectionSlider(options=[1, 2, 4, 8, 16, 32, 64, 128, 256], value=16, description='microsteps', **wide), 'm')
accel = ctl(w.IntSlider(value=600, min=10, max=5000, step=10, description='accel rpm/s', **wide), 'a')
irun  = ctl(w.IntSlider(value=500, min=0, max=700, step=10, description='run mA', **wide), 'i')  # motor rated 670 mA
ihold = ctl(w.IntSlider(value=250, min=0, max=700, step=10, description='hold mA', **wide), 'h')
chop  = ctl(w.ToggleButtons(options=[('stealthChop', 1), ('spreadCycle', 0)], value=1, description='chopper', style={'description_width': '110px'}), 's')
en    = ctl(w.ToggleButtons(options=[('enabled', 1), ('off (freewheel)', 0)], value=1, description='driver', style={'description_width': '110px'}), 'e')

stop = w.Button(description='STOP', button_style='danger')
stop.on_click(lambda _: setattr(rpm, 'value', 0))
refresh = w.Button(description='status')
refresh.on_click(lambda _: run('?'))

# push the slider values so firmware and UI start in the same state (rpm last, after the settings)
for widget, letter in controls[1:] + controls[:1]:
    send(f'{letter}{widget.value:g}')
show(send('?'))

display(w.VBox([rpm, mstep, accel, irun, ihold, chop, en, w.HBox([stop, refresh]), status]))

In [ ]:
# Only when done: stops the motor and frees the port for flashing (left commented so "Run All" does not disconnect)
# rpm.value = 0; time.sleep(1); ser.close()